# 02 — Urban-form inputs for the thermal proof-of-concept

The full Sevilla network is much larger than the initial thermal simulation domain. This notebook therefore creates an approximately 2 km × 2 km proof-of-concept area around the city-centre centroid and prepares a 5 m VoxCity model.

The resulting gridded layers provide the initial:

- building heights;
- canopy heights;
- land-cover classes;
- terrain surface.

At this stage, many building heights are still default values. Notebook 03 replaces them with EUBUCCO heights where available.

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import json
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from voxcity.generator import get_voxcity

boundary = gpd.read_file(DATA_RAW / "sevilla_boundary.gpkg")

## 1. Define the PoC thermal domain

In [ ]:
boundary_projected = boundary.to_crs(boundary.estimate_utm_crs())
centroid = boundary_projected.geometry.centroid.iloc[0]

# A 1 km radius envelope gives an approximately 2 km × 2 km rectangle.
poc_projected = gpd.GeoDataFrame(
    geometry=[centroid.buffer(1000).envelope],
    crs=boundary_projected.crs,
)

poc_wgs84 = poc_projected.to_crs(4326)

ax = poc_wgs84.boundary.plot(figsize=(7, 7))
ax.set_title("Initial Sevilla thermal proof-of-concept domain")
ax.set_axis_off()
plt.show()

## 2. Build the 5 m VoxCity model

In [ ]:
minx, miny, maxx, maxy = poc_wgs84.total_bounds

rectangle_vertices = [
    (minx, miny),
    (minx, maxy),
    (maxx, maxy),
    (maxx, miny),
]

voxcity_model = get_voxcity(
    rectangle_vertices,
    meshsize=5,
)

ds = voxcity_model.to_xarray()
print(ds)

## 3. Inspect building and vegetation inputs

In [ ]:
buildings_vox = voxcity_model.extras["building_gdf"].copy()
trees_vox = voxcity_model.extras["tree_gdf"].copy()

default_height_share = (
    buildings_vox["height_source"].eq("default").mean() * 100
)

print(f"Buildings using default height: {default_height_share:.1f}%")
print("\nBuilding height summary:")
display(buildings_vox["height"].describe())

print("\nTree attributes:")
display(
    trees_vox[
        ["top_height", "bottom_height", "crown_diameter"]
    ].describe()
)

## 4. Visualize the initial gridded urban form

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(voxcity_model.buildings.heights, origin="lower")
axes[0].set_title("Initial building height")

axes[1].imshow(voxcity_model.tree_canopy.top, origin="lower")
axes[1].set_title("Canopy height")

axes[2].imshow(voxcity_model.land_cover.classes, origin="lower")
axes[2].set_title("Land cover")

for ax in axes:
    ax.set_axis_off()

plt.tight_layout()
plt.show()

## 5. Save the baseline urban-form products

In [ ]:
# NetCDF attributes must be serializable; store vertices as JSON text.
ds_to_save = ds.copy()
ds_to_save.attrs["rectangle_vertices"] = json.dumps(
    np.asarray(ds.attrs["rectangle_vertices"]).tolist()
)

ds_to_save.to_netcdf(
    DATA_INTERIM / "sevilla_voxcity_initial.nc"
)

buildings_vox.to_file(
    DATA_INTERIM / "sevilla_buildings_voxcity.gpkg",
    driver="GPKG",
)

trees_vox.to_file(
    DATA_INTERIM / "sevilla_trees_voxcity.gpkg",
    driver="GPKG",
)

metadata = {
    "study_area": "Sevilla thermal PoC",
    "meshsize_m": float(ds.attrs["meshsize"]),
    "crs": str(ds.attrs["crs"]),
    "default_building_height_share": float(
        buildings_vox["height_source"].eq("default").mean()
    ),
}

with open(DATA_INTERIM / "sevilla_voxcity_metadata.json", "w") as handle:
    json.dump(metadata, handle, indent=2)

print("Saved baseline VoxCity inputs.")